# Exploring Eigenvector-Rotation EWS — arXiv:2607.11935
**ArXivist-generated exploratory notebook**

This notebook visualizes the paper's central mechanism: how the TVP-Kalman
elasticity β(t) behaves differently from classical AR(1)/variance EWS, both
on synthetic regional climate data and on the six simulated tipping-point
systems. All plots use the repo's synthetic-fallback data / illustrative
simulation parameters — see `reproduce_arxiv_2607_11935.ipynb` and
`README.md` for details on what's assumption vs. paper-stated fact.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ews_kalman.kalman import TVPKalmanFilter
from ews_kalman.ews import ClassicalEWS
from ews_kalman.data import AIRSDataLoader
from ews_kalman.simulation import TippingSystemSimulator
from ews_kalman.evaluation import SimulationValidator

plt.rcParams["figure.figsize"] = (10, 4)


## 1. β vs classical EWS across the three synthetic regions

Reproduces the qualitative shape of Figure 1: β should show visibly
different dynamics from AR(1) and mutual information (MI) in each region,
with the Tropics region's β sitting close to the Clausius-Clapeyron scaling
value (~0.5) and staying stable, while the Arctic's β is noisier and
weaker.

In [ ]:
loader = AIRSDataLoader()
ews = ClassicalEWS()
kf = TVPKalmanFilter(R=0.001, Q_diag=(1e-6, 1e-7, 1e-8, 1e-9), dt=1/12, mode="loglog")

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=False)
regions = ["arctic", "tropics", "monsoon"]

for ax, region_name in zip(axes, regions):
    region = loader.load_region(region_name, data_dir="/nonexistent/path", seed=0)
    beta_result = kf.estimate_beta(region["T"], region["q"])
    beta = beta_result["beta"]
    ar1_T = ews.rolling_ar1(region["T"], window=24)
    ar1_T_padded = np.concatenate([np.full(len(beta) - len(ar1_T), ar1_T[0]), ar1_T])

    def normalize(x):
        return (x - np.mean(x)) / (np.std(x) + 1e-12)

    ax.plot(normalize(beta), label="beta (TVP-Kalman, normalized)", linewidth=2)
    ax.plot(normalize(ar1_T_padded), label="AR(1) T (normalized)", alpha=0.6)
    ax.set_title(region_name.capitalize())
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Month index")
fig.suptitle("beta vs AR(1), by synthetic region (cf. paper Figure 1)", y=1.0)
plt.tight_layout()
plt.show()


## 2. Six simulated tipping-point systems: β vs AR(1) around the transition

Reproduces the qualitative shape of Figure 3: for coupling-degradation
systems (Stommel AMOC, critical slowing down), β should show a clear
deviation *before* AR(1) does. For the fold bifurcation (where the coupling
itself is unchanged), β should show little to no lead.

In [ ]:
sim = TippingSystemSimulator()
kf_sim = TVPKalmanFilter(R=1e-3, Q_diag=(1e-4, 1e-5, 1e-6, 1e-7), dt=1.0, mode="loglog")
kf_linear = TVPKalmanFilter(R=1e-3, Q_diag=(1e-4, 1e-5, 1e-6, 1e-7), dt=1.0, mode="linear")


def make_positive(s):
    m = np.min(s)
    return s - m + 1.0 if m <= 0 else s


fig, axes = plt.subplots(2, 3, figsize=(15, 7))

# Stommel AMOC (coupling-degradation system -- beta should lead clearly)
amoc = sim.stommel_amoc(seed=0)
beta_amoc = kf_sim.estimate_beta(make_positive(amoc["T"]), make_positive(amoc["S"]))["beta"]
ar1_amoc = ews.rolling_ar1(amoc["T"], window=20)
ar1_amoc_padded = np.concatenate([np.full(len(beta_amoc) - len(ar1_amoc), ar1_amoc[0]), ar1_amoc])

axes[0, 0].plot(beta_amoc, color="tab:green", label="beta")
axes[0, 0].axvline(amoc["tipping_index"], color="black", linestyle="--")
axes[0, 0].set_title("Stommel AMOC: beta")
axes[0, 1].plot(ar1_amoc_padded, color="tab:orange", label="AR(1)")
axes[0, 1].axvline(amoc["tipping_index"], color="black", linestyle="--")
axes[0, 1].set_title("Stommel AMOC: AR(1)")
axes[0, 2].plot(amoc["T"], label="T (box 1)", alpha=0.7)
axes[0, 2].plot(amoc["S"], label="S (box 2)", alpha=0.7)
axes[0, 2].axvline(amoc["tipping_index"], color="black", linestyle="--")
axes[0, 2].set_title("Stommel AMOC: raw driver/response")
axes[0, 2].legend(fontsize=7)

# Fold bifurcation (coupling itself unchanged -- beta should NOT lead)
fold = sim.fold_bifurcation(seed=0)
x_pos = make_positive(fold["x"])
y_pos = np.roll(x_pos, -1); y_pos[-1] = y_pos[-2]
beta_fold = kf_sim.estimate_beta(x_pos, y_pos)["beta"]
ar1_fold = ews.rolling_ar1(fold["x"], window=20)
ar1_fold_padded = np.concatenate([np.full(len(beta_fold) - len(ar1_fold), ar1_fold[0]), ar1_fold])

axes[1, 0].plot(beta_fold, color="tab:green")
axes[1, 0].axvline(fold["tipping_index"], color="black", linestyle="--")
axes[1, 0].set_title("Fold bifurcation: beta (should NOT lead)")
axes[1, 1].plot(ar1_fold_padded, color="tab:orange")
axes[1, 1].axvline(fold["tipping_index"], color="black", linestyle="--")
axes[1, 1].set_title("Fold bifurcation: AR(1)")
axes[1, 2].plot(fold["x"], color="tab:blue")
axes[1, 2].axvline(fold["tipping_index"], color="black", linestyle="--")
axes[1, 2].set_title("Fold bifurcation: raw x")

plt.tight_layout()
plt.show()

print("Interpretation (paper Section 3.4): beta leads clearly for Stommel AMOC")
print("(a coupling-degradation system) but not for the fold bifurcation, where")
print("the coupling y~x^2 itself is unchanged -- this is the paper's key")
print("specificity result: beta detects coupling changes, not univariate")
print("stability changes.")


## 3. Table 3 summary: lead-time bar chart

Visualizes the lead-time comparison across all six systems in one view.

In [ ]:
validator = SimulationValidator()
results = validator.validate_all_systems(seed=0)

names = [r["simulation"] for r in results]
beta_leads = [r["beta_lead"] if r["beta_lead"] is not None else 0 for r in results]
ar1_leads = [r["ar1_lead"] if r["ar1_lead"] is not None else 0 for r in results]

x_pos = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x_pos - width/2, beta_leads, width, label="beta lead", color="tab:green")
ax.bar(x_pos + width/2, ar1_leads, width, label="AR(1) lead", color="tab:orange")
ax.set_xticks(x_pos)
ax.set_xticklabels(names, rotation=20, ha="right")
ax.set_ylabel("Lead time (timesteps before tipping point)")
ax.set_title("Table 3 reproduction: beta vs AR(1) lead time per simulated system")
ax.legend()
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

for r in results:
    print(f"{r['simulation']:<22} winner={r['winner']}")


## Next steps

- Swap the synthetic AIRS fallback for real Giovanni-portal data
  (see `data/README_data.md`) to reproduce the paper's exact Table 1/2 numbers.
- Tune `configs/config.yaml`'s `simulations` block to try to match Table 3's
  exact lead-time values (noise magnitudes and forcing schedules are not
  given numerically in the paper — see README.md "Reproducibility Notes").
- Run `python run_all.py --config configs/config.yaml --output-dir results/`
  for the full end-to-end reproduction with saved Figures 1-3.
